# O Problema
Sliding Puzzle - Bloco Deslizante

In [24]:
# !wget -qq https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif
from IPython.display import Image
Image(url='https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif',width=200)

Roteiro de estudo
1. Construir uma estimativa
Abra a explicação interativa do Red Blob Games e acompanhe a comparação entre busca em largura, busca gulosa e A*. Observe quais posições cada estratégia analisa antes de chegar ao destino.

A função que estima o custo restante é chamada de heurística, escrita como h(n). No 8-puzzle, vamos usar a distância de Manhattan: para cada peça numerada, conte quantas linhas e colunas a separam da posição correta e some os valores. O espaço vazio não entra nessa conta.

Por exemplo, se apenas a peça 8 estiver ao lado de sua posição final e puder entrar nela com uma jogada, a estimativa será 1. Em outros tabuleiros, as peças podem atrapalhar umas às outras: somar as distâncias não resolve o quebra-cabeça, apenas fornece um limite inferior para o número de movimentos necessários.

2. Comparar duas maneiras de usar a pista
A busca gulosa escolhe o nó com menor h(n). Ela dá prioridade ao que parece mais próximo do objetivo. Isso pode reduzir bastante a exploração, mas também pode conduzir a um caminho ruim. Ela não garante o caminho mais curto nem é sempre mais rápida.

O A* considera também o custo já percorrido, g(n). Sua prioridade é f(n) = g(n) + h(n). Se um nó tem g=8 e h=2, sua prioridade é 10. Outro com g=3 e h=4 tem prioridade 7. A gulosa escolheria o primeiro; A* escolheria o segundo.

Acompanhe agora a parte de buscas informadas da aula “Search”, do CS50. Refaça uma comparação de prioridades antes de conferir a escolha do algoritmo. Os slides da aula podem ser usados depois como revisão.

3. Entender a garantia do A*
Para encontrar um caminho ótimo, não basta que a heurística pareça razoável. Uma heurística admissível nunca estima um custo maior que o mínimo realmente necessário. A distância de Manhattan tem essa propriedade no 8-puzzle com movimentos de custo 1: cada jogada desloca uma única peça numerada em uma casa.

Ela também é consistente: a estimativa antes de uma jogada não é maior que o custo dessa jogada somado à estimativa depois dela. Com essa propriedade e o tratamento correto dos custos, A* pode fechar um estado quando o retira da fila de prioridade, sem precisar expandi-lo novamente. A implementação deve reconhecer a solução quando o objetivo for retirado da fila, e não assim que ele aparecer entre os sucessores.

O capítulo de Russell e Norvig é a referência para aprofundar essas condições. Se você experimentar outra heurística, não presuma que ela mantém as mesmas garantias.

#Estrutura:

In [25]:
class No:
    def __init__(self, estado, no_pai=None, acao=None, custo_acumulado=0):
      self.estado = estado
      self.no_pai = no_pai
      self.acao = acao
      self.custo_acumulado = custo_acumulado

    def __repr__(self):
      return f"No(estado={self.estado}, acao={self.acao}, custo={self.custo_acumulado})"

estado_inicial = ((1, 2, 3), (4, 5, 6), (7, 0, 8)) #0 é esp. vazio

no_raiz = No(estado=estado_inicial)
print("Nó raiz:")
display(no_raiz)

novo_estado = ((1, 2, 3), (4, 5, 6), (0, 7, 8)) #moveu p direita
no_sucessor = No(estado=novo_estado, no_pai=no_raiz, acao="mover_para_direita", custo_acumulado=no_raiz.custo_acumulado + 1)
print("\nNó sucessor:")
display(no_sucessor)

no_pai = no_sucessor.no_pai
print("\nNó pai do nó sucessor (nó raiz):")
display(no_pai)

Nó raiz:


No(estado=((1, 2, 3), (4, 5, 6), (7, 0, 8)), acao=None, custo=0)


Nó sucessor:


No(estado=((1, 2, 3), (4, 5, 6), (0, 7, 8)), acao=mover_para_direita, custo=1)


Nó pai do nó sucessor (nó raiz):


No(estado=((1, 2, 3), (4, 5, 6), (7, 0, 8)), acao=None, custo=0)

In [26]:
import collections

# Definir o estado objetivo para um puzzle 3x3
estado_objetivo = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

def encontrar_posicao_vazia(estado):
    """Encontra a posição (linha, coluna) do espaço vazio (0) no estado."""
    for r in range(3):
        for c in range(3):
            if estado[r][c] == 0:
                return r, c
    return None

def obter_acoes_validas(estado):
    """Retorna uma lista de ações válidas ('cima', 'baixo', 'esquerda', 'direita')
       para o estado dado, com base na posição do espaço vazio."""
    r_vazio, c_vazio = encontrar_posicao_vazia(estado)
    acoes = []
    # Mover para cima (blank moves down)
    if r_vazio > 0: acoes.append('cima')
    # Mover para baixo (blank moves up)
    if r_vazio < 2: acoes.append('baixo')
    # Mover para esquerda (blank moves right)
    if c_vazio > 0: acoes.append('esquerda')
    # Mover para direita (blank moves left)
    if c_vazio < 2: acoes.append('direita')
    return acoes

def aplicar_acao(estado, acao):
    """Retorna o novo estado após aplicar a ação dada."""
    r_vazio, c_vazio = encontrar_posicao_vazia(estado)
    novo_estado = [list(row) for row in estado] # Converter para lista de listas para mutabilidade

    # Coordenadas para troca
    r_troca, c_troca = r_vazio, c_vazio
    if acao == 'cima': r_troca -= 1
    elif acao == 'baixo': r_troca += 1
    elif acao == 'esquerda': c_troca -= 1
    elif acao == 'direita': c_troca += 1

    # Realizar a troca
    novo_estado[r_vazio][c_vazio], novo_estado[r_troca][c_troca] = \
        novo_estado[r_troca][c_troca], novo_estado[r_vazio][c_vazio]

    return tuple(tuple(row) for row in novo_estado) # Converter de volta para tupla de tuplas

def reconstruir_caminho(no_final):
    """Reconstrói o caminho da solução do nó final até o nó raiz."""
    caminho = []
    no_atual = no_final
    while no_atual.no_pai is not None:
        caminho.append(no_atual.acao)
        no_atual = no_atual.no_pai
    return caminho[::-1] # Inverter a lista para obter o caminho do início ao fim

# Resolver o quebra-cabeças usando Buscas

## Busca em largura (BFS)

In [27]:
def bfs(estado_inicial, estado_objetivo):
  fronteira = collections.deque()
  no_raiz = No(estado_inicial)
  fronteira.append(no_raiz)

  visitados= {estado_inicial}

  while fronteira:
    no_atual = fronteira.popleft()
    if no_atual.estado == estado_objetivo:
      return reconstruir_caminho(no_atual), no_atual.custo_acumulado

    #expande o nó atual
    for acao in obter_acoes_validas(no_atual.estado):
      proximo_estado = aplicar_acao(no_atual.estado, acao)
      #se prox estado nao foi visitado, adiciona a fronteira e marca como visitado
      if proximo_estado not in visitados:
        visitados.add(proximo_estado)
        proximo_no = No(proximo_estado, no_pai=no_atual, acao=acao, custo_acumulado=no_atual.custo_acumulado + 1)
        fronteira.append(proximo_no)

  return None, None

### Exemplo de Uso do BFS

In [28]:
# Definir um estado inicial diferente (apenas para demonstração)
estado_teste_inicial = ((1, 2, 3), (4, 0, 6), (7, 5, 8))

print(f"Estado Inicial para o teste:\n{estado_teste_inicial}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho, custo_solucao = bfs(estado_teste_inicial, estado_objetivo)

if solucao_caminho:
    print(f"\nSolução encontrada em {custo_solucao} passos:")
    print(solucao_caminho)
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido.")


Estado Inicial para o teste:
((1, 2, 3), (4, 0, 6), (7, 5, 8))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução encontrada em 2 passos:
['baixo', 'direita']


## Busca em Profundidade (DFS)

In [29]:
def dfs(estado_inicial, estado_objetivo):
    # fronteira se comporta como uma pilha Last In, First Out
    # por isso usaremos uma lista Python e os métodos append/pop
    fronteira = []
    no_raiz = No(estado_inicial)
    fronteira.append(no_raiz)

    visitados = {estado_inicial} # estados visitados

    while fronteira:
        no_atual = fronteira.pop()

        # Verifica se é o objetivo
        if no_atual.estado == estado_objetivo:
            return reconstruir_caminho(no_atual), no_atual.custo_acumulado

        # Expande o nó
        for acao in obter_acoes_validas(no_atual.estado):
            proximo_estado = aplicar_acao(no_atual.estado, acao)

            # Se o próximo estado não foi visitado, adiciona-o à fronteira e marca como visitado
            if proximo_estado not in visitados:
                visitados.add(proximo_estado)
                proximo_no = No(proximo_estado, no_pai=no_atual, acao=acao, custo_acumulado=no_atual.custo_acumulado + 1)
                fronteira.append(proximo_no)

    return None, None # Se nenhuma solução for encontrada




### Exemplo de Uso do DFS

In [30]:

estado_teste_inicial_dfs = ((1, 2, 3), (4, 0, 6), (7, 5, 8))

print(f"Estado Inicial para o teste:\n{estado_teste_inicial_dfs}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho_dfs, custo_solucao_dfs = dfs(estado_teste_inicial_dfs, estado_objetivo)

if solucao_caminho_dfs:
    print(f"\nSolução encontrada em {custo_solucao_dfs} passos:")
    print(solucao_caminho_dfs)
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido com DFS.")

Estado Inicial para o teste:
((1, 2, 3), (4, 0, 6), (7, 5, 8))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução encontrada em 49284 passos:
['direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'cima', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'cima', 'esquerda', 'esquerda', 'baixo', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'cima', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', '

##Atividade: Resolvendo o 8-Puzzle com A*
A atividade usa o mesmo 8-puzzle da semana 4. Aproveite a representação do tabuleiro e a geração de movimentos para implementar A* e comparar os resultados. Se ainda não concluiu a atividade anterior, comece conferindo essas duas partes com um tabuleiro a uma jogada do objetivo.

O tabuleiro é uma grade 3×3 com oito peças numeradas e um espaço vazio (0). O objetivo é ordená-lo até o estado alvo:

1 2 3\
4 5 6\
7 8 0\
Componentes a implementar (no Colab):

- Representação do estado: escolha a estrutura que preferir (lista, matriz, tupla…).
- Nó de busca: deve guardar: o estado, g(n), h(n), f(n), o nó pai e o movimento que o gerou (para reconstruir o caminho no final).
- Geração de sucessores: localize o 0, identifique os movimentos válidos (cima/baixo/esquerda/direita, sem sair do tabuleiro) e gere os novos estados.
- Custo g(n): número de movimentos desde o início; cada movimento custa 1, então g(filho) = g(pai) + 1.
- Heurística h(n): use a Distância de Manhattan: para cada peça, |linha_atual - linha_objetivo| + |coluna_atual - coluna_objetivo|, somada para todas as peças (o espaço vazio 0 não entra no cálculo).
- Avaliação f(n) = g(n) + h(n): lembre-se: o A* expande o estado com menor f(n), não o de menor h(n). Essa é a diferença para a Busca Gulosa.
Use uma fila de prioridade para a fronteira e guarde o menor custo g encontrado para cada estado. Se descobrir um caminho mais barato até um estado que ainda está na fronteira, atualize o custo e o caminho correspondente. Se a fila guardar entradas antigas, descarte-as quando forem retiradas.

Não marque um estado como encerrado só porque ele foi gerado. Com Manhattan neste problema, o conjunto de estados fechados deve registrar os que já foram retirados para expansão pelo menor f. Heurísticas admissíveis que não sejam consistentes podem exigir reabrir estados quando surgir um caminho melhor.

Ao final, exiba:

- O estado final resolvido e o caminho (sequência de movimentos) encontrado;
- O número de movimentos da solução;
- A quantidade de estados expandidos;
- (Opcional) O tempo de execução.

Compare BFS, DFS e A* nos mesmos tabuleiros. Comece por casos pequenos, em que a BFS consiga servir de referência para o menor número de movimentos. Registre também como desempata prioridades iguais; essa escolha pode alterar a quantidade de nós expandidos.

Antes de seguir, confira: você consegue calcular g, h e f de um nó e explicar qual deles a gulosa usa? Seu A* devolve um caminho válido com o mesmo comprimento da BFS nesses casos?

Desafio Extra (Opcional): Solubilidade
Nem toda configuração do 8-Puzzle tem solução: algumas nunca alcançam o objetivo, faça o que fizer. Antes de rodar o A*, implemente uma verificação que detecte se o estado inicial é solucionável e, se não for, avise e encerre. Pesquise sobre número de inversões e paridade de estados.

Reflexão: qual a vantagem de checar a solubilidade antes de rodar o A*? O que acontece com o tempo de execução ao tentar resolver um estado insolúvel? E em versões maiores, como o 15-Puzzle?

In [31]:
import heapq
import time # For optional execution time tracking

def heuristica_manhattan(estado_atual, estado_objetivo):
    distancia = 0
    posicoes_objetivo = {}
    for r in range(3):
        for c in range(3):
            peca = estado_objetivo[r][c]
            if peca != 0: # Ignorar o espaço vazio
                posicoes_objetivo[peca] = (r, c)

    for r_atual in range(3):
        for c_atual in range(3):
            peca = estado_atual[r_atual][c_atual]
            if peca != 0: # Ignorar o espaço vazio
                r_objetivo, c_objetivo = posicoes_objetivo[peca]
                distancia += abs(r_atual - r_objetivo) + abs(c_atual - c_objetivo) #distancia de cada peça a posicao objetivo
    return distancia #retorna a soma das distancias de manhattan de todas as peças



---

#Busca Gulosa

In [32]:
def busca_gulosa(estado_inicial, estado_objetivo):
    start_time = time.time()

    # busca gulosa prioriza sò o valor da heurística h(n)
    fronteira = []

    # armazenar estados já visitados p evitar ciclos
    visitados = set()

    nos_para_estado = {}

    no_raiz = No(estado_inicial)
    h_inicial = heuristica_manhattan(estado_inicial, estado_objetivo)

    heapq.heappush(fronteira, (h_inicial, estado_inicial))
    nos_para_estado[estado_inicial] = no_raiz
    visitados.add(estado_inicial)

    estados_expandidos_count = 0

    while fronteira:
        h_atual, estado_atual_tupla = heapq.heappop(fronteira)
        no_atual = nos_para_estado[estado_atual_tupla]

        # estado atual é o objetivo?
        if estado_atual_tupla == estado_objetivo:
            end_time = time.time()
            tempo_execucao = end_time - start_time
            return reconstruir_caminho(no_atual), no_atual.custo_acumulado, estados_expandidos_count, tempo_execucao

        estados_expandidos_count += 1

        # Expande o nó atual
        for acao in obter_acoes_validas(estado_atual_tupla):
            proximo_estado = aplicar_acao(estado_atual_tupla, acao)

            # Se o próximo estado não foi visitado, adiciona ele a fronteira
            if proximo_estado not in visitados:
                visitados.add(proximo_estado)
                nova_h = heuristica_manhattan(proximo_estado, estado_objetivo)
                novo_g = no_atual.custo_acumulado + 1 # atualiza o custo g

                proximo_no = No(proximo_estado, no_pai=no_atual, acao=acao, custo_acumulado=novo_g)
                nos_para_estado[proximo_estado] = proximo_no
                heapq.heappush(fronteira, (nova_h, proximo_estado))

    end_time = time.time()
    tempo_execucao = end_time - start_time
    return None, None, estados_expandidos_count, tempo_execucao # Nenhuma solução encontrada

print("Algoritmo de Busca Gulosa definido.")

Algoritmo de Busca Gulosa definido.


### Exemplo de Uso da Busca Gulosa

In [37]:
estado_teste_inicial_gulosa = ((1, 2, 3), (4, 0, 6), (7, 5, 8))

print(f"Estado Inicial para o teste Gulosa:\n{estado_teste_inicial_gulosa}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho_gulosa, custo_solucao_gulosa, estados_expandidos_gulosa, tempo_gulosa = busca_gulosa(estado_teste_inicial_gulosa, estado_objetivo)

if solucao_caminho_gulosa:
    print(f"\nSolução encontrada em {custo_solucao_gulosa} passos.")
    print(f"Caminho: {solucao_caminho_gulosa}")
    print(f"Estados expandidos: {estados_expandidos_gulosa}")
    print(f"Tempo de execução: {tempo_gulosa:.4f} segundos")
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido com Busca Gulosa.")

print("\n--- Teste com estado mais complexo ---")
# Exemplo de estado que pode exigir mais passos
estado_complexo_inicial_gulosa = ((8, 6, 7), (2, 5, 4), (3, 0, 1))

print(f"Estado Inicial:\n{estado_complexo_inicial_gulosa}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho_complexo_gulosa, custo_solucao_complexo_gulosa, estados_expandidos_complexo_gulosa, tempo_complexo_gulosa = busca_gulosa(estado_complexo_inicial_gulosa, estado_objetivo)

if solucao_caminho_complexo_gulosa:
    print(f"\nSolução encontrada em {custo_solucao_complexo_gulosa} passos.")
    print(f"Caminho: {solucao_caminho_complexo_gulosa}")
    print(f"Estados expandidos: {estados_expandidos_complexo_gulosa}")
    print(f"Tempo de execução: {tempo_complexo_gulosa:.4f} segundos")
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido com Busca Gulosa (complexo).")

Estado Inicial para o teste Gulosa:
((1, 2, 3), (4, 0, 6), (7, 5, 8))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução encontrada em 2 passos.
Caminho: ['baixo', 'direita']
Estados expandidos: 2
Tempo de execução: 0.0001 segundos

--- Teste com estado mais complexo ---
Estado Inicial:
((8, 6, 7), (2, 5, 4), (3, 0, 1))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução encontrada em 53 passos.
Caminho: ['esquerda', 'cima', 'cima', 'direita', 'direita', 'baixo', 'baixo', 'esquerda', 'esquerda', 'cima', 'cima', 'direita', 'direita', 'baixo', 'baixo', 'esquerda', 'esquerda', 'cima', 'cima', 'direita', 'direita', 'baixo', 'baixo', 'esquerda', 'esquerda', 'cima', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'baixo', 'direita', 'cima', 'esquerda', 'cima', 'direita', 'baixo', 'baixo', 'esquerda', 'cima', 'direita', 'cima', 'esquerda', 'baixo', 'baixo', 'direita', 'cima', 'esquerda', 'cima', 'direita', 'baixo', 'baixo']
Estados expandidos: 92
Tempo de execução: 0.0016 



---
#A*


In [34]:

def astar(estado_inicial, estado_objetivo):
    start_time = time.time()

    # Fila de prioridade: (f_score, h_score, g_score, estado) para desempate
    # f_score: g + h
    # h_score: heurística de Manhattan
    # g_score: custo do caminho do início até o nó atual
    # estado: a representação do tabuleiro
    fronteira = []

    # custos_g_atingidos armazena o menor custo g encontrado para cada estado
    custos_g_atingidos = {estado_inicial: 0}

    # mapeia cada estado para o No que representa a melhor forma de chegar a esse estado até o momento
    nos_para_estado = {}

    no_raiz = No(estado_inicial)
    h_inicial = heuristica_manhattan(estado_inicial, estado_objetivo)
    f_inicial = h_inicial # g_inicial é 0 para o nó raiz

    heapq.heappush(fronteira, (f_inicial, h_inicial, no_raiz.custo_acumulado, estado_inicial))
    nos_para_estado[estado_inicial] = no_raiz

    estados_expandidos_count = 0

    while fronteira:
        f_atual, h_atual, g_atual, estado_atual_tupla = heapq.heappop(fronteira)

        # Se o custo g que acabamos de retirar da fila é maior do que o menor custo g já registrado
        # para este estado, significa que já encontramos um caminho melhor. Descartar esta entrada antiga.
        if g_atual > custos_g_atingidos.get(estado_atual_tupla, float('inf')):
            continue

        no_atual = nos_para_estado[estado_atual_tupla]

        # Verifica se o estado atual é o objetivo
        if estado_atual_tupla == estado_objetivo:
            end_time = time.time()
            tempo_execucao = end_time - start_time
            return reconstruir_caminho(no_atual), no_atual.custo_acumulado, estados_expandidos_count, tempo_execucao

        estados_expandidos_count += 1

        # Expande o nó atual
        for acao in obter_acoes_validas(estado_atual_tupla):
            proximo_estado = aplicar_acao(estado_atual_tupla, acao)
            novo_g = no_atual.custo_acumulado + 1 # Custo de cada movimento é 1

            # Se encontrarmos um caminho mais curto para este 'proximo_estado'
            if novo_g < custos_g_atingidos.get(proximo_estado, float('inf')):
                custos_g_atingidos[proximo_estado] = novo_g
                nova_h = heuristica_manhattan(proximo_estado, estado_objetivo)
                novo_f = novo_g + nova_h

                proximo_no = No(proximo_estado, no_pai=no_atual, acao=acao, custo_acumulado=novo_g)
                nos_para_estado[proximo_estado] = proximo_no # Atualiza ou adiciona o nó com o melhor caminho
                heapq.heappush(fronteira, (novo_f, nova_h, novo_g, proximo_estado))

    end_time = time.time()
    tempo_execucao = end_time - start_time
    return None, None, estados_expandidos_count, tempo_execucao # Nenhuma solução encontrada




## Exemplo de Uso do A*

In [35]:

estado_teste_inicial_astar = ((1, 2, 3), (4, 0, 6), (7, 5, 8))

print(f"Estado Inicial para o teste A*:\n{estado_teste_inicial_astar}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho_astar, custo_solucao_astar, estados_expandidos_astar, tempo_astar = astar(estado_teste_inicial_astar, estado_objetivo)

if solucao_caminho_astar:
    print(f"\nSolução A* encontrada em {custo_solucao_astar} passos.")
    print(f"Caminho: {solucao_caminho_astar}")
    print(f"Estados expandidos: {estados_expandidos_astar}")
    print(f"Tempo de execução: {tempo_astar:.4f} segundos")
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido com A*.")

print("\n--- Teste com um estado mais complexo ---")
estado_complexo_inicial = ((8, 6, 7), (2, 5, 4), (3, 0, 1))

print(f"Estado Inicial:\n{estado_complexo_inicial}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho_complexo, custo_solucao_complexo, estados_expandidos_complexo, tempo_complexo = astar(estado_complexo_inicial, estado_objetivo)

if solucao_caminho_complexo:
    print(f"\nSolução encontrada em {custo_solucao_complexo} passos.")
    print(f"Caminho: {solucao_caminho_complexo}")
    print(f"Estados expandidos: {estados_expandidos_complexo}")
    print(f"Tempo de execução: {tempo_complexo:.4f} segundos")
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido com A* (complexo).")

Estado Inicial para o teste A*:
((1, 2, 3), (4, 0, 6), (7, 5, 8))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução A* encontrada em 2 passos.
Caminho: ['baixo', 'direita']
Estados expandidos: 2
Tempo de execução: 0.0002 segundos

--- Teste com um estado mais complexo ---
Estado Inicial para o teste A* (complexo):
((8, 6, 7), (2, 5, 4), (3, 0, 1))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução A* encontrada em 31 passos.
Caminho: ['esquerda', 'cima', 'direita', 'baixo', 'direita', 'cima', 'cima', 'esquerda', 'baixo', 'esquerda', 'baixo', 'direita', 'direita', 'cima', 'cima', 'esquerda', 'esquerda', 'baixo', 'baixo', 'direita', 'direita', 'cima', 'esquerda', 'cima', 'esquerda', 'baixo', 'baixo', 'direita', 'cima', 'direita', 'baixo']
Estados expandidos: 6843
Tempo de execução: 0.2752 segundos
